# LayoutLMv1 Inference Example

In [15]:
!sudo apt install -y tesseract-ocr
!pip install  pytesseract transformers datasets
! pip install tesseract-ocr

Reading package lists... Done
Building dependency tree       
Reading state information... Done
E: Unable to locate package tesseract-ocr
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 54.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tesseract-ocr (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [45 lines of output]
      /tmp/pip-build-env-r6839yol/overlay/lib/python3.12/site-packages/setuptools/_distutils/dist.py:289: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      /tmp/pip-build-env-r6839yol/overlay/lib/python3.12/site-packages/setuptools/dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              *******

In [18]:
!sudo apt update -y
!sudo apt install -y tesseract-ocr

Hit:1 https://packages.microsoft.com/repos/microsoft-ubuntu-focal-prod focal InRelease
Hit:2 https://dl.yarnpkg.com/debian stable InRelease                           
Hit:3 https://repo.anaconda.com/pkgs/misc/debrepo/conda stable InRelease       
Hit:4 http://archive.ubuntu.com/ubuntu focal InRelease                         
Hit:5 http://security.ubuntu.com/ubuntu focal-security InRelease           
Hit:6 http://archive.ubuntu.com/ubuntu focal-updates InRelease       
Hit:7 http://archive.ubuntu.com/ubuntu focal-backports InRelease     m
Hit:8 https://packagecloud.io/github/git-lfs/ubuntu focal InRelease  
Reading package lists... Done
Building dependency tree       
Reading state information... Done
46 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following additional packages will be installed:
  liblept5 libopenjp2-7 libtesseract4 libwebpmux3 tesseract-ocr-eng
  t

## Token Classification/Object Detection

In [19]:
from transformers import LayoutLMForTokenClassification, LayoutLMv2Processor
from PIL import Image, ImageDraw, ImageFont
import torch

# load model and processor from huggingface hub
model = LayoutLMForTokenClassification.from_pretrained("philschmid/layoutlm-funsd")
processor = LayoutLMv2Processor.from_pretrained("philschmid/layoutlm-funsd")


# helper function to unnormalize bboxes for drawing onto the image
def unnormalize_box(bbox, width, height):
    return [
        width * (bbox[0] / 1000),
        height * (bbox[1] / 1000),
        width * (bbox[2] / 1000),
        height * (bbox[3] / 1000),
    ]


label2color = {
    "B-HEADER": "blue",
    "B-QUESTION": "red",
    "B-ANSWER": "green",
    "I-HEADER": "blue",
    "I-QUESTION": "red",
    "I-ANSWER": "green",
}
# draw results onto the image
def draw_boxes(image, boxes, predictions):
    width, height = image.size
    normalizes_boxes = [unnormalize_box(box, width, height) for box in boxes]

    # draw predictions over the image
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    for prediction, box in zip(predictions, normalizes_boxes):
        if prediction == "O":
            continue
        draw.rectangle(box, outline="black")
        draw.rectangle(box, outline=label2color[prediction])
        draw.text((box[0] + 10, box[1] - 10), text=prediction, fill=label2color[prediction], font=font)
    return image


# run inference
def run_inference(path, model=model, processor=processor, output_image=True):
    # create model input
    image = Image.open(path).convert("RGB")
    encoding = processor(image, return_tensors="pt")
    del encoding["image"]
    # run inference
    outputs = model(**encoding)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    # get labels
    labels = [model.config.id2label[prediction] for prediction in predictions]
    if output_image:
        return draw_boxes(image, encoding["bbox"][0], labels)
    else:
        return labels



In [24]:
pwd

'/workspaces/document-ai-transformers'

In [ ]:
run_inference("/workspaces/document-ai-transformers/inference/invoice_example.jpg")


FileNotFoundError: [Errno 2] No such file or directory: 'workspaces/document-ai-transformers/inference/invoice_example.jpg'